In [1]:
"""
Read a custom INCAR and check whether LOCPROJ is parsed correctly
"""

'\nRead a custom INCAR and check whether LOCPROJ is parsed correctly\n'

In [2]:
from collections import Counter, UserDict

from monty.io import zopen
from monty.json import MontyDecoder, MSONable

import os
import orjson

import re
from collections.abc import Mapping, Sequence

from typing import Any, ClassVar, Literal, Self
from pymatgen.util.string import str_delimited
import tabulate
from pymatgen.io.vasp import Incar
import importlib
import pymatgen
# importlib.reload(pymatgen.io.vasp.inputs)

import warnings

MODULE_DIR = "./"


In [3]:
import pymatgen.io.vasp.inputs
print(pymatgen.io.vasp.inputs.__file__)

/Users/ar/venvs/pmg_dev/lib/python3.14/site-packages/pymatgen/io/vasp/inputs.py


In [4]:
# class IncarFix(UserDict, MSONable):
#     """
#     A case-insensitive dictionary to read/write INCAR files with additional helper functions.

#     - Keys are stored in uppercase to allow case-insensitive access (set, get, del, update, setdefault).
#     - String values are capitalized by default, except for keys specified
#         in the `lower_str_keys/as_is_str_keys` of the `proc_val` method.
#     """

#     # INCAR tag/value recording
#     with open(os.path.join(MODULE_DIR, "incar_parameters.json"), encoding="utf-8") as json_file:
#         INCAR_PARAMS: ClassVar[dict[Literal["type", "values"], Any]] = orjson.loads(json_file.read())

#     LOCPROJ_YLM_SPECS = [
#         "s",
#         "p",     "py", "pz", "px",
#         "d",     "dxy", "dyz", "dz2", "dxz", "dx2-y2",
#         "f",     "fy(3x2-y2)", "fxyz", "fyz2", "fz3", "fxz2", "fz(x2-y2)", "fx(x2-3y2)",
#         "sp",    "sp-1", "sp-2",
#         "sp2",   "sp2-1", "sp2-2", "sp2-3",
#         "sp3",   "sp3-1", "sp3-2", "sp3-3", "sp3-4",
#         "sp3d",  "sp3d-1", "sp3d-2", "sp3d-3", "sp3d-4", "sp3d-5",
#         "sp3d2", "sp3d2-1", "sp3d2-2", "sp3d2-3", "sp3d2-4", "sp3d2-5", "sp3d2-6"
#     ]

#     LOCPROJ_RADIAL_SPECS = [
#         "Ps", "Pr", "Hy"
#     ]

#     def __init__(self, params: Mapping[str, Any] | None = None) -> None:
#         """
#         Clean up params and create an Incar object.

#         Args:
#             params (dict): INCAR parameters as a dictionary.

#         Warnings:
#             BadIncarWarning: If there are duplicate in keys (case insensitive).
#         """
#         params = params or {}

#         # Check for case-insensitive duplicate keys
#         key_counter = Counter(key.strip().upper() for key in params)
#         if duplicates := [key for key, count in key_counter.items() if count > 1]:
#             warnings.warn(
#                 f"Duplicate keys found (case-insensitive): {duplicates}",
#                 BadIncarWarning,
#                 stacklevel=2,
#             )

#         # If INCAR contains vector-like MAGMOMS given as list[float], convert to list[list]
#         if (params.get("MAGMOM") and isinstance(params["MAGMOM"][0], int | float)) and (
#             params.get("LSORBIT") or params.get("LNONCOLLINEAR")
#         ):
#             params["MAGMOM"] = [params["MAGMOM"][idx * 3 : (idx + 1) * 3] for idx in range(len(params["MAGMOM"]) // 3)]
#         elif (params.get("LOCPROJ")):
#             # parse locproj to a good format
#             locproj = params["LOCPROJ"]
#             if isinstance(locproj, str):
#                 locproj_parsed = self.parse_locproj(locproj)
#             elif isinstance(locproj, list) and all(isinstance(x, str) for x in locproj):
#                 locproj_parsed = []
#                 for entry in locproj:
#                     locproj_parsed.extend(self.parse_locproj(entry))
#             elif isinstance(locproj, list) and all(isinstance(x, list) and len(x) == 3 for x in locproj):
#                 locproj_parsed = []
#                 for entry in locproj:
#                     idx, orb, rad_ = entry
#                     if orb.lower() not in [y.lower() for y in self.LOCPROJ_YLM_SPECS]:
#                         warnings.warn(
#                             f"LOCPROJ orbital '{orb}' is not in recognized options: {self.LOCPROJ_YLM_SPECS}.",
#                             UserWarning,
#                             stacklevel=2,
#                         )
#                     if rad_ not in self.LOCPROJ_RADIAL_SPECS:
#                         warnings.warn(
#                             f"LOCPROJ radial '{rad_}' is not in recognized options: {self.LOCPROJ_RADIAL_SPECS}.",
#                             UserWarning,
#                             stacklevel=2,
#                         )
#                     locproj_parsed.append(entry)
                
#             else:
#                 raise ValueError("LOCPROJ is not in a recognized format.")
#             print("Sanitized LOCPROJ:", locproj_parsed)
#             params["LOCPROJ"] = locproj_parsed

        

#         super().__init__(params)

#     def __setitem__(self, key: str, val: Any) -> None:
#         """
#         Add parameter-val pair to Incar.
#         - Clean the parameter and val by stripping leading
#             and trailing white spaces.
#         - Cast keys to upper case.
#         """
#         key = key.strip().upper()
#         # Cast float/int to str such that proc_val would clean up their types
#         val = self.proc_val(key, str(val)) if isinstance(val, str | float | int) else val
#         super().__setitem__(key, val)

#     def __getitem__(self, key: str) -> Any:
#         """
#         Get value using a case-insensitive key.
#         """
#         return super().__getitem__(key.strip().upper())

#     def __delitem__(self, key: str) -> None:
#         super().__delitem__(key.strip().upper())

#     def __contains__(self, key: str) -> bool:
#         return super().__contains__(key.upper().strip())

#     def __str__(self) -> str:
#         return self.get_str(sort_keys=True, pretty=False)

#     def __add__(self, other: Self) -> Self:
#         """
#         Add all the values of another INCAR object to this object.
#         Facilitate the use of "standard" INCARs.
#         """
#         params: dict[str, Any] = dict(self.items())
#         for key, val in other.items():
#             if key in self and val != self[key]:
#                 raise ValueError(f"INCARs have conflicting values for {key}: {self[key]} != {val}")
#             params[key] = val
#         return type(self)(params)

#     def get(self, key: str, default: Any = None) -> Any:
#         """
#         Get a value for a case-insensitive key, return default if not found.
#         """
#         return super().get(key.strip().upper(), default)

#     def as_dict(self) -> dict:
#         """MSONable dict."""
#         dct = dict(self)
#         dct["@module"] = type(self).__module__
#         dct["@class"] = type(self).__name__
#         return dct

#     @classmethod
#     def _validate_locproj_entry(cls, orb: str, rad: str, stacklevel: int = 3) -> None:
#         """Warning for when LOCPROJ specs are not matching the allowed values"""
#         if orb.lower() not in [y.lower() for y in cls.LOCPROJ_YLM_SPECS]:
#             warnings.warn(
#                 f"LOCPROJ orbital '{orb}' is not in recognized options: {cls.LOCPROJ_YLM_SPECS}.",
#                 UserWarning,
#                 stacklevel=stacklevel,
#             )
#         if rad not in cls.LOCPROJ_RADIAL_SPECS:
#             warnings.warn(
#                 f"LOCPROJ radial '{rad}' is not in recognized options: {cls.LOCPROJ_RADIAL_SPECS}.",
#                 UserWarning,
#                 stacklevel=stacklevel,
#             )

#     @classmethod
#     def from_dict(cls, dct: dict[str, Any]) -> Self:
#         """
#         Args:
#             dct (dict): Serialized Incar.

#         Returns:
#             Incar
#         """
#         if dct.get("MAGMOM") and isinstance(dct["MAGMOM"][0], dict):
#             dct["MAGMOM"] = [Magmom.from_dict(m) for m in dct["MAGMOM"]]
#         return cls({k: v for k, v in dct.items() if k not in ("@module", "@class")})

#     def copy(self) -> Self:
#         return type(self)(self)  # type:ignore[arg-type]

#     def get_str(self, sort_keys: bool = False, pretty: bool = False) -> str:
#         """Get a string representation of the INCAR. Differ from the
#         __str__ method to provide options for pretty printing.

#         Args:
#             sort_keys (bool): Whether to sort the INCAR parameters
#                 alphabetically. Defaults to False.
#             pretty (bool): Whether to pretty align output.
#                 Defaults to False.
#         """
#         keys: list[str] = sorted(self) if sort_keys else list(self)
#         lines = []
#         for key in keys:

                
#             if key == "MAGMOM" and isinstance(self[key], list):
#                 value = []

#                 if isinstance(self[key][0], list | Magmom) and (self.get("LSORBIT") or self.get("LNONCOLLINEAR")):
#                     value.append(" ".join(str(i) for j in self[key] for i in j))
#                 elif self.get("LSORBIT") or self.get("LNONCOLLINEAR"):
#                     for _key, group in itertools.groupby(self[key]):
#                         value.append(f"3*{len(tuple(group))}*{_key}")
#                 else:
#                     # float() to ensure backwards compatibility between
#                     # float magmoms and Magmom objects
#                     for _key, group in itertools.groupby(self[key], key=float):
#                         value.append(f"{len(tuple(group))}*{_key}")

#                 lines.append([key, " ".join(value)])


#             # print(key == "LOCPROJ")
#             elif key == "LOCPROJ" and isinstance(self[key], list):
#                 locproj_inside = "; ".join(self[key])
#                 locproj_string = f"\"{locproj_inside}\"" # format of "1 : s : Hy; 2: p : Hy; ..." as required by VASP

#                 lines.append([key, locproj_string])
#             elif isinstance(self[key], list):
#                 lines.append([key, " ".join(map(str, self[key]))])
#             else:
#                 lines.append([key, self[key]])
                

#         if pretty:
#             return str(tabulate([[line[0], "=", line[1]] for line in lines], tablefmt="plain"))
#         return str_delimited(lines, None, " = ") + "\n"

#     def write_file(self, filename: PathLike) -> None:
#         """Write Incar to a file.

#         Args:
#             filename (str): filename to write to.
#         """
#         with zopen(filename, mode="wt", encoding="utf-8") as file:
#             file.write(str(self))  # type:ignore[arg-type]

#     @classmethod
#     def parse_locproj(cls, val):
#         ylm = cls.LOCPROJ_YLM_SPECS
#         rad = cls.LOCPROJ_RADIAL_SPECS

#         # Strip inline comments (! or #)
#         val = re.sub(r"[!#].*", "", val)

#         # Each entry must end at ; or end-of-string, not bleed into next
#         pattern = re.compile(
#             r"(\d+(?:\s+\d+)*)\s*:\s*([^:;\s]+)\s*:\s*([^:;\s]+)\s*(?:;|$)",
#             re.IGNORECASE
#         )
#         results = []
#         for match in pattern.finditer(val):
#             indices = [int(i) for i in match.group(1).split()]
#             orb = match.group(2)
#             rad_ = match.group(3)

#             if orb.lower() not in [y.lower() for y in ylm]:
#                 warnings.warn(
#                     f"LOCPROJ orbital '{orb}' is not in recognized options: {ylm}.",
#                     UserWarning,
#                     stacklevel=3,
#                 )

#             if rad_ not in rad:
#                 warnings.warn(
#                     f"LOCPROJ radial '{rad_}' is not in recognized options: {rad}.",
#                     UserWarning,
#                     stacklevel=3,
#                 )

#             for idx in indices:
#                 results.append([idx, orb, rad_])
#         return results

#     @classmethod
#     def from_file(cls, filename: PathLike) -> Self:
#         """Read an Incar object from a file.

#         Args:
#             filename (str): Filename for file

#         Returns:
#             Incar object
#         """
#         with zopen(filename, mode="rt", encoding="utf-8") as file:
#             return cls.from_str(file.read())  # type:ignore[arg-type]

#     @classmethod
#     def from_str(cls, string: str) -> Self:
#         """Read an Incar object from a string.

#         Args:
#             string (str): Incar string

#         Returns:
#             Incar object
#         """
#         string = "\n".join([ln.split("#", 1)[0].split("!", 1)[0].rstrip() for ln in string.splitlines()])

#         params: dict[str, Any] = {}

#         # Handle line continuations (\)
#         string = re.sub(r"\\\s*\n", " ", string)

#         # Regex pattern to find all valid "key = value" assignments at once
#         pattern = re.compile(
#             r"""
#             (?P<key>\w+)             # Key (e.g. ENCUT)
#             \s*=\s*                  # Equals sign and optional spaces
#             (?:                      # Non-capturing group for the value
#                 "                    # Opening quote
#                 (?P<qval>.*?)        # Capture everything inside (non-greedy)
#                 [ \t]*"              # Allow trailing spaces/tabs before closing quote
#                 |                    # OR
#                 (?P<val>[^#!;\n]*)   # Unquoted value (stops before comment/separator)
#             )
#             """,
#             re.VERBOSE | re.DOTALL,
#         )

#         # Find all matches in the entire string
#         for match in pattern.finditer(string):
#             key = match.group("key")
#             val = match.group("qval") if match.group("qval") is not None else (match.group("val") or "").strip()

#             if not val:
#                 continue

#             params[key] = cls.proc_val(key, val)

#         return cls(params)

#     @classmethod
#     def proc_val(cls, key: str, val: str) -> list | bool | float | int | str:
#         """Helper method to convert INCAR parameters to proper types
#         like ints, floats, lists, etc.

#         Args:
#             key (str): INCAR parameter key.
#             val (str): Value of INCAR parameter.
#         """
#         # Handle union type (e.g. "bool | str" for LREAL)
#         if incar_type := cls.INCAR_PARAMS.get(key, {}).get("type"):
#             incar_types: list[str] = [t.strip() for t in incar_type.split("|")]
#         else:
#             incar_types = []

#         # Special cases
#         # Always lower case
#         lower_str_keys = ("ML_MODE",)
#         # String keywords to read "as is" (no case transformation, only stripped)
#         as_is_str_keys = ("SYSTEM", "WANNIER90_WIN")
#         multiline_str_keys = ("LOCPROJ",) # <-- Modified

#         try:
#             if key in lower_str_keys:
#                 return val.strip().lower()

#             if key in as_is_str_keys:
#                 return val.strip()
            
#             if key in multiline_str_keys:
#                 # really just LOCPROJ tag
#                 return cls.parse_locproj(val)
#                 # pattern = re.compile(r"(\d+(?:\s+\d+)*)\s*:\s*(" + "|".join(map(re.escape, cls.LOCPROJ_YLM_SPECS)) + r")\s*:\s*(" + "|".join(map(re.escape, cls.LOCPROJ_RADIAL_SPECS)) + r")", re.IGNORECASE)
#                 # results = []
#                 # for match in pattern.finditer(val):
#                 #     indices = [int(i) for i in match.group(1).split()]
#                 #     orb = match.group(2)
#                 #     rad = match.group(3)
#                 #     for idx in indices:
#                 #         results.append([idx, orb, rad])
#                 # if not results:
#                 #     warnings.warn(f"LOCPROJ value could not be parsed: '{val}'", BadIncarWarning, stacklevel=2)
                    
#                 # return results

#             if "list" in incar_types:

#                 def smart_int_or_float_bool(str_: str) -> float | int | bool:
#                     """Determine whether a string represents an integer or a float."""
#                     if str_.lower().startswith(".t") or str_.lower().startswith("t"):
#                         return True
#                     if str_.lower().startswith(".f") or str_.lower().startswith("f"):
#                         return False
#                     if "." in str_ or "e" in str_.lower():
#                         return float(str_)
#                     return int(str_)

#                 output = []
#                 tokens = re.findall(r"(-?\d+\.?\d*|[\.A-Z]+)\*?(-?\d+\.?\d*|[\.A-Z]+)?\*?(-?\d+\.?\d*|[\.A-Z]+)?", val)
#                 for tok in tokens:
#                     if tok[2] and "3" in tok[0]:
#                         output.extend([smart_int_or_float_bool(tok[2])] * int(tok[0]) * int(tok[1]))
#                     elif tok[1]:
#                         output.extend([smart_int_or_float_bool(tok[1])] * int(tok[0]))
#                     else:
#                         output.append(smart_int_or_float_bool(tok[0]))

#                 if output:  # pass when fail to parse (val is not list)
#                     return output

#             if "bool" in incar_types:
#                 if match := re.match(r"^\.?([T|F|t|f])[A-Za-z]*\.?", val):
#                     return match[1].lower() == "t"

#                 raise ValueError(f"{key} should be a boolean type!")

#             if "float" in incar_types:
#                 return float(re.search(r"^-?\d*\.?\d*[e|E]?-?\d*", val)[0])  # type: ignore[index]

#             if "int" in incar_types:
#                 return int(re.match(r"^-?[0-9]+", val)[0])  # type: ignore[index]

#         # If re.match doesn't hit, it would return None and thus TypeError from indexing
#         except (ValueError, TypeError):
#             pass

#         # Not in known keys. We will try a hierarchy of conversions.
#         try:
#             return int(val)
#         except ValueError:
#             pass

#         try:
#             return float(val)
#         except ValueError:
#             pass

#         if "true" in val.lower():
#             return True

#         if "false" in val.lower():
#             return False

#         return val.strip().capitalize()


# class BadIncarWarning(UserWarning):
#     """Warning class for bad INCAR parameters."""

In [5]:
incar1 = Incar.from_file("INCAR1")
incar1

{'ALGO': 'Normal', 'EDIFF': 0.0001, 'ENCUT': 520.0, 'ICHARG': 11, 'ISMEAR': -5, 'ISPIN': 2, 'ISYM': 2, 'LASPH': True, 'LCHARG': False, 'LMAXMIX': 4, 'LOCPROJ': [1, 2, 1, 1, 1], 'LORBIT': 11, 'LREAL': 'Auto', 'LWAVE': False, 'NEDOS': 2001, 'NELM': 100, 'NSW': 0, 'PREC': 'Accurate', 'SIGMA': 0.05}

In [ ]:
print(incar1['LOCPROJ'])
# Sanity check and parse LOCPROJ if needed
# def parse_locproj(val):
#     import re
#     ylm = IncarFix.LOCPROJ_YLM_SPECS
#     rad = IncarFix.LOCPROJ_RADIAL_SPECS
#     pattern = re.compile(r"(\d+(?:\s+\d+)*)\s*:\s*([^:;\s]+)\s*:\s*([^:;\s]+)", re.IGNORECASE)
#     results = []
#     for match in pattern.finditer(val):
#         indices = [int(i) for i in match.group(1).split()]
#         orb = match.group(2)
#         rad_ = match.group(3)
#         for idx in indices:
#             results.append([idx, orb, rad_])
#     return results

# locproj = incar1['LOCPROJ']
# if isinstance(locproj, str):
#     locproj_parsed = parse_locproj(locproj)
# elif isinstance(locproj, list) and all(isinstance(x, str) for x in locproj):
#     locproj_parsed = []
#     for entry in locproj:
#         locproj_parsed.extend(parse_locproj(entry))
# elif isinstance(locproj, list) and all(isinstance(x, list) and len(x) == 3 for x in locproj):
#     locproj_parsed = locproj
# else:
#     raise ValueError("LOCPROJ is not in a recognized format.")
# print("Sanitized LOCPROJ:", locproj_parsed)
incar_fix = IncarFix({"LOCPROJ": [[1, 's','Ps'], [1,'r','Py']]})

incar_fix
# incar_fix.write_file("INCAR_fix")

NameError: name 'IncarFix' is not defined

In [ ]:
"1 : s: Hy; 1 : s : Hy"

[[1, 's', 'Hy'], [1, 's', 'Hy']]

In [ ]:
with zopen("INCAR1", mode="rt", encoding="utf-8") as file:
    f = file.read()
    string = "\n".join([ln.split("#", 1)[0].split("!", 1)[0].rstrip() for ln in f.splitlines()])
    params: dict[str, Any] = {}
    params_raw: dict[str, Any] = {}

    string = re.sub(r"\\\s*\n", " ", string)

    # Regex pattern to find all valid "key = value" assignments at once
    pattern = re.compile(
        r"""
        (?P<key>\w+)             # Key (e.g. ENCUT)
        \s*=\s*                  # Equals sign and optional spaces
        (?:                      # Non-capturing group for the value
            "                    # Opening quote
            (?P<qval>.*?)        # Capture everything inside (non-greedy)
            [ \t]*"              # Allow trailing spaces/tabs before closing quote
            |                    # OR
            (?P<val>[^#!;\n]*)   # Unquoted value (stops before comment/separator)
        )
        """,
        re.VERBOSE | re.DOTALL,
    )

    # # Find all matches in the entire string
    for match in pattern.finditer(string):
        key = match.group("key")
        val = match.group("qval") if match.group("qval") is not None else (match.group("val") or "").strip()

        if not val:
            continue

        params_raw[key] =  val
        params[key] = IncarFix.proc_val(key, val)



'1 : d : Ps'

In [ ]:
current_val = params_raw["LOCPROJ"]
output = []
tokens = re.findall(r"(-?\d+\.?\d*|[\.A-Z]+)\*?(-?\d+\.?\d*|[\.A-Z]+)?\*?(-?\d+\.?\d*|[\.A-Z]+)?", current_val)
for tok in tokens:
    if tok[2] and "3" in tok[0]:
        output.extend([my_smart_int_or_float_bool(tok[2])] * int(tok[0]) * int(tok[1]))
    elif tok[1]:
        output.extend([my_smart_int_or_float_bool(tok[1])] * int(tok[0]))
    else:
        output.append(my_smart_int_or_float_bool(tok[0]))

output

ValueError: invalid literal for int() with base 10: 'P'